# Aletheia: Sentiment Analysis Training Notebook

Aletheia is a sentiment analysis classifier that reads between the lines. In this notebook, we load the IMDB dataset, explore the reviews, train a Logistic Regression model with TF-IDF features, evaluate it, and test it on custom sentences.

## 1. Introduction

We use TF-IDF plus Logistic Regression because it is a strong, fast baseline for text classification. TF-IDF turns text into numbers by giving more weight to important words and less weight to common ones. Logistic Regression then learns which word patterns usually map to positive or negative sentiment.

In [ ]:
# Import the libraries we need for data loading, modeling, and plotting
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# Try to import WordCloud for the EDA section. If it is missing, the notebook still runs.
try:
    from wordcloud import WordCloud
except ImportError:
    WordCloud = None

plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path('..').resolve()
RANDOM_STATE = 42

## 2. Data loading

The IMDB dataset from Hugging Face contains 50,000 labeled movie reviews. Each review is already marked as positive or negative, which makes it ideal for supervised learning.

In [ ]:
# Load the IMDB dataset from Hugging Face
dataset = load_dataset('imdb')

# Convert the train and test splits into one DataFrame for exploration
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()
data = pd.concat([train_df, test_df], ignore_index=True)
data = data.rename(columns={'text': 'review', 'label': 'sentiment'})

# Map labels to readable names
data['sentiment_label'] = data['sentiment'].map({0: 'Negative', 1: 'Positive'})
data.head()

## 3. Exploratory Data Analysis

Before training, it helps to understand the shape of the data. Here we look at class balance, review length, and the most common words in the dataset.

In [ ]:
# Class distribution
class_counts = data['sentiment_label'].value_counts()
print('Class distribution:')
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(class_counts.index, class_counts.values, color=['#c92a2a', '#1f8f4c'])
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# Review length distribution
review_lengths = data['review'].str.split().str.len()
axes[1].hist(review_lengths, bins=40, color='#4c78a8', edgecolor='white')
axes[1].set_title('Review Length Distribution')
axes[1].set_xlabel('Number of words')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Build a simple word cloud from a sample of positive and negative reviews
sample_text = ' '.join(data['review'].sample(n=3000, random_state=RANDOM_STATE).astype(str))
if WordCloud is not None:
    wordcloud = WordCloud(width=1200, height=600, background_color='white', max_words=150).generate(sample_text)
    plt.figure(figsize=(14, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud')
    plt.show()
else:
    print('Install the wordcloud package to render the word cloud visualization.')

## 4. Data preprocessing

TF-IDF converts text into a sparse matrix of numbers. Each row is a review and each column is a word or phrase. Words that appear often in one review but are rare across the whole dataset get higher weight, which helps the model focus on meaningful terms instead of common filler words.

In [ ]:
# Split the data into features and labels
X = data['review']
y = data['sentiment']

# Keep the split reproducible and stratified so both classes stay balanced
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# TF-IDF turns text into numeric features that Logistic Regression can learn from
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print('Training matrix shape:', X_train_tfidf.shape)
print('Test matrix shape:', X_test_tfidf.shape)

## 5. Model training

Now we train a Logistic Regression model on the TF-IDF features. This model is a good fit for text classification because it is simple, fast, and strong enough to serve as a reliable baseline.

In [ ]:
# Create the classifier
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)

# Train the model on the TF-IDF features
model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print(f'F1 score: {f1:.4f}')

## 6. Evaluation

A confusion matrix shows how many reviews were classified correctly and where the model made mistakes. The classification report adds precision, recall, and F1 score for each class.

In [ ]:
# Build and display the confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Negative', 'Positive'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['Negative', 'Positive'])

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, cm[row, col], ha='center', va='center', color='black')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Print the classification report for a fuller evaluation summary
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 7. Inference

Finally, we test the trained model on a few custom sentences. This shows how the model behaves on new text that was not part of training.

In [ ]:
# Try a few custom examples
custom_sentences = [
    'This movie was surprisingly moving and beautifully acted.',
    'I was bored the entire time and wanted it to end quickly.',
    'The plot was fine, but the ending felt rushed and uneven.'
]

custom_tfidf = vectorizer.transform(custom_sentences)
custom_predictions = model.predict(custom_tfidf)
custom_probabilities = model.predict_proba(custom_tfidf)

for sentence, prediction, probabilities in zip(custom_sentences, custom_predictions, custom_probabilities):
    label = 'Positive' if int(prediction) == 1 else 'Negative'
    confidence = float(max(probabilities) * 100)
    print(f'Text: {sentence}')
    print(f'Prediction: {label} ({confidence:.1f}% confidence)')
    print('-' * 80)